# Naïve Bayes — Theory & Implementation

This notebook provides a comprehensive walkthrough of the **Naïve Bayes** classifier:

1. **Bayes' Theorem** — mathematical foundation
2. **Naïve Bayes assumption** — conditional independence & formula derivation
3. **Manual calculation** — lookup-table approach
4. **From-scratch implementation** — a full `NaiveBayesClassifier` class
5. **Types of Naïve Bayes** — Gaussian, Multinomial, Bernoulli
6. **Laplace Smoothing** — handling the zero-frequency problem
7. **Pros, Cons & When to Use**

> **Companion notebook:** [`03-sentiment-analysis-naive-bayes.ipynb`](03-sentiment-analysis-naive-bayes.ipynb) — IMDB sentiment analysis with Naïve Bayes.

---
## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries imported successfully!")

---
## 2. Bayes' Theorem — Mathematical Foundation

Bayes' theorem describes the probability of an event, based on prior knowledge of conditions related to the event.

$$P(A \mid B) = \frac{P(B \mid A) \cdot P(A)}{P(B)}$$

Where:
- $P(A \mid B)$ — **Posterior probability**: probability of hypothesis $A$ given the observed evidence $B$
- $P(B \mid A)$ — **Likelihood**: probability of evidence $B$ given that hypothesis $A$ is true
- $P(A)$ — **Prior probability**: probability of hypothesis $A$ before observing evidence
- $P(B)$ — **Marginal likelihood (Evidence)**: total probability of the evidence

### Example: Play Tennis Dataset

We use a classic dataset where weather conditions determine whether someone plays tennis.

In [ ]:
df_tennis = pd.read_csv("../../datasets/playtennis.csv")
print(f"Dataset shape: {df_tennis.shape}")
df_tennis

In [ ]:
# --- Bayes' Theorem: Code Example ---

def bayes_theorem(p_a, p_b_given_a, p_b):
    """Compute P(A|B) using Bayes' theorem."""
    return (p_b_given_a * p_a) / p_b


# What is P(Yes | Outlook=Sunny)?
total = len(df_tennis)
p_yes = len(df_tennis[df_tennis['PlayTennis'] == 'Yes']) / total
p_no  = len(df_tennis[df_tennis['PlayTennis'] == 'No'])  / total

p_sunny = len(df_tennis[df_tennis['Outlook'] == 'Sunny']) / total
p_sunny_given_yes = len(df_tennis[(df_tennis['Outlook'] == 'Sunny') &
                                   (df_tennis['PlayTennis'] == 'Yes')]) / \
                    len(df_tennis[df_tennis['PlayTennis'] == 'Yes'])
p_sunny_given_no  = len(df_tennis[(df_tennis['Outlook'] == 'Sunny') &
                                   (df_tennis['PlayTennis'] == 'No')]) / \
                    len(df_tennis[df_tennis['PlayTennis'] == 'No'])

posterior_yes = bayes_theorem(p_yes, p_sunny_given_yes, p_sunny)
posterior_no  = bayes_theorem(p_no,  p_sunny_given_no,  p_sunny)

print("=== Bayes' Theorem Example: P(PlayTennis | Outlook=Sunny) ===")
print(f"P(Yes)         = {p_yes:.4f}")
print(f"P(No)          = {p_no:.4f}")
print(f"P(Sunny)       = {p_sunny:.4f}")
print(f"P(Sunny | Yes) = {p_sunny_given_yes:.4f}")
print(f"P(Sunny | No)  = {p_sunny_given_no:.4f}")
print(f"---")
print(f"P(Yes | Sunny) = {posterior_yes:.4f}")
print(f"P(No  | Sunny) = {posterior_no:.4f}")
print(f"Prediction: {'Yes' if posterior_yes > posterior_no else 'No'}")

---
## 3. Naïve Bayes Assumption and Formula Derivation

### The Conditional Independence Assumption

Naïve Bayes makes a **strong (naïve) assumption**: all features are **conditionally independent** given the class label.

$$P(x_1, x_2, \ldots, x_n \mid y) = \prod_{i=1}^{n} P(x_i \mid y)$$

### Full Naïve Bayes Formula

$$P(y \mid x_1, x_2, \ldots, x_n) \propto P(y) \cdot \prod_{i=1}^{n} P(x_i \mid y)$$

The predicted class is:
$$\hat{y} = \arg\max_{y} \; P(y) \cdot \prod_{i=1}^{n} P(x_i \mid y)$$

### Worked Example

Classify: **X = (Outlook=Sunny, Temperature=Cool, Humidity=High, Wind=Strong)**

In [ ]:
# Step-by-step Naive Bayes calculation for:
# X = (Outlook=Sunny, Temperature=Cool, Humidity=High, Wind=Strong)

new_sample = {
    'Outlook': 'Sunny', 'Temperature': 'Cool',
    'Humidity': 'High', 'Wind': 'Strong'
}

features = ['Outlook', 'Temperature', 'Humidity', 'Wind']
classes = df_tennis['PlayTennis'].unique()

print("=" * 60)
print(f"Classifying: {new_sample}")
print("=" * 60)

posteriors = {}
for cls in classes:
    prior = len(df_tennis[df_tennis['PlayTennis'] == cls]) / total
    print(f"\n--- Class: {cls} ---")
    print(f"  Prior P({cls}) = {prior:.4f}")

    likelihood = 1.0
    for feat in features:
        cls_subset = df_tennis[df_tennis['PlayTennis'] == cls]
        p_feat = len(cls_subset[cls_subset[feat] == new_sample[feat]]) / len(cls_subset)
        print(f"  P({feat}={new_sample[feat]} | {cls}) = {p_feat:.4f}")
        likelihood *= p_feat

    posteriors[cls] = prior * likelihood
    print(f"  Unnormalized posterior = {posteriors[cls]:.6f}")

total_post = sum(posteriors.values())
print(f"\n{'=' * 60}")
print("Normalized Posteriors:")
for cls, val in posteriors.items():
    print(f"  P({cls} | X) = {val / total_post:.4f}" if total_post > 0 else "  N/A")
print(f"\n>>> Predicted class: {max(posteriors, key=posteriors.get)}")

---
## 4. Load and Explore the Sample Dataset

Let's explore the Play Tennis dataset systematically.

In [ ]:
print("First 5 rows:")
df_tennis.head()

In [ ]:
print("Dataset Info:")
df_tennis.info()

In [ ]:
print("\nDataset Description:")
df_tennis.describe()

In [ ]:
# Class distribution
print("Class Distribution (PlayTennis):")
print(df_tennis['PlayTennis'].value_counts())
print()

for col in ['Outlook', 'Temperature', 'Humidity', 'Wind']:
    print(f"\n{col} distribution:")
    print(df_tennis[col].value_counts())

In [ ]:
# Visualize feature distributions
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for idx, col in enumerate(['PlayTennis', 'Outlook', 'Temperature', 'Humidity', 'Wind']):
    df_tennis[col].value_counts().plot(kind='bar', ax=axes[idx], color='steelblue', edgecolor='black')
    axes[idx].set_title(col)
    axes[idx].set_ylabel('Count')
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.suptitle('Feature Distributions — Play Tennis Dataset', y=1.02, fontsize=14)
plt.show()

---
## 5. Manual Naïve Bayes Calculation (Lookup Table Approach)

We build **frequency tables** and **likelihood tables** manually using pandas, then make a prediction.

In [ ]:
# Prior probabilities
class_counts = df_tennis['PlayTennis'].value_counts()
priors = class_counts / len(df_tennis)
print("Prior Probabilities:")
print(priors)
print()

# Conditional probability tables for each feature
likelihood_tables = {}

for feat in features:
    print(f"{'='*50}")
    print(f"Feature: {feat}")
    print(f"{'='*50}")

    freq_table = pd.crosstab(df_tennis[feat], df_tennis['PlayTennis'])
    print("\nFrequency Table:")
    print(freq_table)

    likelihood_table = freq_table.div(freq_table.sum(axis=0), axis=1)
    print("\nLikelihood Table P(feature | class):")
    print(likelihood_table.round(4))
    print()

    likelihood_tables[feat] = likelihood_table

In [ ]:
# Predict using lookup tables
print(f"New Sample: {new_sample}")
print("=" * 50)

results = {}
for cls in ['Yes', 'No']:
    prob = priors[cls]
    print(f"\nClass '{cls}':")
    print(f"  Prior P({cls}) = {prob:.4f}")

    for feat in features:
        cond_prob = likelihood_tables[feat].loc[new_sample[feat], cls]
        print(f"  P({feat}={new_sample[feat]} | {cls}) = {cond_prob:.4f}")
        prob *= cond_prob

    results[cls] = prob
    print(f"  => Unnormalized posterior = {prob:.6f}")

total_prob = sum(results.values())
print(f"\n{'='*50}")
print("Final Normalized Posteriors:")
for cls, val in results.items():
    print(f"  P({cls} | X) = {val/total_prob:.4f}")
print(f"\n>>> Prediction: PlayTennis = {max(results, key=results.get)}")

---
## 6. Implement Naïve Bayes From Scratch

A full `NaiveBayesClassifier` class for **categorical features**, with `fit()` and `predict()` methods.

In [ ]:
class NaiveBayesClassifier:
    """Naive Bayes classifier for categorical features with optional Laplace smoothing."""

    def __init__(self, alpha=0):
        self.alpha = alpha
        self.class_priors = {}
        self.likelihoods = {}
        self.classes = None
        self.features = None
        self.feature_values = {}

    def fit(self, X, y):
        self.classes = y.unique()
        self.features = X.columns.tolist()
        n_total = len(y)

        for cls in self.classes:
            self.class_priors[cls] = np.sum(y == cls) / n_total

        for feat in self.features:
            self.feature_values[feat] = X[feat].unique()
            self.likelihoods[feat] = {}

            for feat_val in self.feature_values[feat]:
                self.likelihoods[feat][feat_val] = {}

                for cls in self.classes:
                    cls_mask = (y == cls)
                    n_cls = np.sum(cls_mask)
                    n_feat_cls = np.sum((X[feat] == feat_val) & cls_mask)
                    n_unique = len(self.feature_values[feat])

                    self.likelihoods[feat][feat_val][cls] = \
                        (n_feat_cls + self.alpha) / (n_cls + self.alpha * n_unique)

        print("Model fitted successfully!")
        print(f"  Classes: {list(self.classes)}")
        print(f"  Priors: {self.class_priors}")
        return self

    def _predict_single(self, x):
        posteriors = {}
        for cls in self.classes:
            posterior = self.class_priors[cls]
            for feat in self.features:
                feat_val = x[feat]
                if feat_val in self.likelihoods[feat]:
                    posterior *= self.likelihoods[feat][feat_val].get(cls, self.alpha)
                else:
                    posterior *= self.alpha if self.alpha > 0 else 1e-10
            posteriors[cls] = posterior
        return max(posteriors, key=posteriors.get), posteriors

    def predict(self, X):
        return [self._predict_single(X.iloc[i])[0] for i in range(len(X))]

    def predict_proba(self, X):
        all_posteriors = []
        for i in range(len(X)):
            _, posteriors = self._predict_single(X.iloc[i])
            total = sum(posteriors.values())
            all_posteriors.append({k: v/total for k, v in posteriors.items()})
        return all_posteriors

In [ ]:
# Test on Play Tennis dataset
X_tennis = df_tennis[features]
y_tennis = df_tennis['PlayTennis']

nb_scratch = NaiveBayesClassifier(alpha=0)
nb_scratch.fit(X_tennis, y_tennis)

train_preds = nb_scratch.predict(X_tennis)
train_acc = accuracy_score(y_tennis, train_preds)
print(f"\nTraining Accuracy: {train_acc:.4f}")

# Predict the new sample
new_df = pd.DataFrame([new_sample])
pred = nb_scratch.predict(new_df)
proba = nb_scratch.predict_proba(new_df)
print(f"\nNew sample prediction: {pred[0]}")
print(f"Posterior probabilities: {proba[0]}")
print("\nThis matches our manual calculation above!")

---
## 7. Types of Naïve Bayes Classifiers

| Variant | Feature Type | Distribution Assumption | Typical Use Case |
|---|---|---|---|
| **GaussianNB** | Continuous | Gaussian (normal) distribution | Iris, general continuous data |
| **MultinomialNB** | Discrete (counts) | Counts or frequencies | Text classification (word counts) |
| **BernoulliNB** | Binary (0/1) | Binary indicators | Text classification (word presence) |

### Gaussian Naïve Bayes — PDF Calculation

For continuous features, GaussianNB uses the **probability density function** of the Gaussian distribution:

$$P(x_i \mid y) = \frac{1}{\sqrt{2\pi\sigma_y^2}} \exp\left(-\frac{(x_i - \mu_y)^2}{2\sigma_y^2}\right)$$

In [ ]:
# Gaussian PDF visualization
def gaussian_pdf(x, mean, std):
    """Compute Gaussian probability density."""
    exponent = np.exp(-((x - mean) ** 2) / (2 * std ** 2))
    return (1 / (np.sqrt(2 * np.pi) * std)) * exponent

x_range = np.linspace(-5, 10, 300)
mean_0, std_0 = 2.0, 1.5
mean_1, std_1 = 6.0, 1.0

plt.figure(figsize=(10, 5))
plt.plot(x_range, gaussian_pdf(x_range, mean_0, std_0), 'b-', lw=2,
         label=f'Class 0 (\u03bc={mean_0}, \u03c3={std_0})')
plt.plot(x_range, gaussian_pdf(x_range, mean_1, std_1), 'r-', lw=2,
         label=f'Class 1 (\u03bc={mean_1}, \u03c3={std_1})')
plt.axvline(x=4.5, color='green', ls='--', label='Decision boundary (approx)')
plt.fill_between(x_range, gaussian_pdf(x_range, mean_0, std_0), alpha=0.2, color='blue')
plt.fill_between(x_range, gaussian_pdf(x_range, mean_1, std_1), alpha=0.2, color='red')
plt.xlabel('Feature Value', fontsize=12)
plt.ylabel('Probability Density', fontsize=12)
plt.title('Gaussian Naive Bayes — Class-Conditional Distributions', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

test_x = 3.0
print(f"\nP(x={test_x} | Class 0) = {gaussian_pdf(test_x, mean_0, std_0):.4f}")
print(f"P(x={test_x} | Class 1) = {gaussian_pdf(test_x, mean_1, std_1):.4f}")
print(f"=> x={test_x} is more likely under Class 0")

In [ ]:
# GaussianNB on the Iris dataset
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42)

gnb = GaussianNB()
gnb.fit(X_train_iris, y_train_iris)
y_pred_iris = gnb.predict(X_test_iris)

print(f"GaussianNB on Iris — Accuracy: {accuracy_score(y_test_iris, y_pred_iris):.4f}")
print(f"\n{classification_report(y_test_iris, y_pred_iris, target_names=iris.target_names)}")

# Confusion matrix
cm = confusion_matrix(y_test_iris, y_pred_iris)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.title('GaussianNB — Iris Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

---
## 8. Laplace Smoothing — The Zero-Frequency Problem

### The Problem

If a feature value **never appears** with a particular class during training:

$$P(x_i \mid y) = 0$$

Since Naïve Bayes **multiplies** all conditional probabilities, **one zero kills the entire product**:

$$P(y \mid x_1, \ldots, x_n) \propto P(y) \times \ldots \times 0 \times \ldots = 0$$

### Laplace (Add-1) Smoothing

$$P(x_i \mid y) = \frac{\text{count}(x_i, y) + \alpha}{\text{count}(y) + \alpha \cdot |V|}$$

Where:
- $\alpha$ = smoothing parameter (typically 1)
- $|V|$ = number of unique values for the feature

---
## 9. Advantages, Disadvantages & When to Use

### Advantages

| # | Advantage |
|---|---|
| 1 | **Extremely fast** training and prediction — $O(n \cdot d)$ time complexity |
| 2 | Works well with **small training datasets** |
| 3 | Handles **high-dimensional data** effectively (e.g., text with thousands of features) |
| 4 | **Simple to implement** and easy to interpret |
| 5 | **Good baseline** for text classification tasks |
| 6 | Not sensitive to **irrelevant features** (they cancel out across classes) |
| 7 | Performs well even when the independence assumption is **partially violated** |

### Disadvantages

| # | Disadvantage |
|---|---|
| 1 | **Strong independence assumption** rarely holds in real-world data |
| 2 | Poor **probability estimates** (posteriors are often too extreme) |
| 3 | **Zero-frequency problem** — requires smoothing to avoid zeros |
| 4 | Cannot learn **feature interactions** (e.g., XOR-like patterns) |
| 5 | Outperformed by more complex models (SVM, Random Forest, Neural Nets) on **large datasets** |
| 6 | Sensitive to **correlated features** — violates the core assumption |

### When to Use Naive Bayes

- **Text classification** — spam detection, sentiment analysis, topic categorization
- **Real-time prediction** — where speed matters more than marginal accuracy gains
- **Small or medium-sized datasets** — where you need a quick, reliable baseline
- **High-dimensional sparse data** — e.g., bag-of-words representations
- As a **benchmark / baseline** to compare against more complex models

---
## Summary

- **Bayes' Theorem** provides the foundation: $P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$
- **Naive Bayes** assumes conditional independence to simplify the joint probability
- **Laplace smoothing** ($\alpha$) prevents zero-probability collapse
- **Three main variants**: GaussianNB (continuous), MultinomialNB (counts), BernoulliNB (binary)
- Despite its simplicity, Naive Bayes remains a **strong baseline** for many classification tasks, especially in NLP

> **Next:** See [`03-sentiment-analysis-naive-bayes.ipynb`](03-sentiment-analysis-naive-bayes.ipynb) for a hands-on sentiment analysis project using these classifiers.